<a href="https://colab.research.google.com/github/GuruNathan24/Tamil-NLP/blob/master/madx_tamil_emotion_colab_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAD-X style zero-shot Tamil emotion classification

English emotion **task adapter** (trained) + Tamil **language adapter** (swapped in at inference). No Tamil emotion labels are used for training, so you still need a labelled Tamil test set to measure real performance.

**Google Colab setup**
1. `Runtime -> Change runtime type -> T4 GPU` (training on CPU would take many hours).
2. Run the install cell. If Colab shows **Restart session**, click it and continue from the next cell (do not re-run the install cell).
3. Keep your labelled Tamil CSV (`text`, `label`) in Google Drive, or upload it when asked in the last section.

In [ ]:
from datasets import load_dataset

# Load the dair-ai/emotion dataset
dataset = load_dataset('dair-ai/emotion')

# Display the dataset details and some samples
print(dataset)
print("\nTrain sample:")
display(dataset['train'][0])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

Train sample:


{'text': 'i didnt feel humiliated', 'label': 0}

In [ ]:
# Colab already has torch, transformers, pandas and scikit-learn.
# `adapters` may change the transformers version. If Colab shows "Restart session", click it,
# then continue from the next cell (do NOT re-run this cell).
%pip install -q adapters datasets accelerate sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.5/295.5 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 123.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import os, random
import numpy as np
import torch

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoTokenizer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
)
from adapters import AutoAdapterModel, AdapterTrainer
from adapters.composition import Stack

import transformers, adapters
print("torch", torch.__version__, "| transformers", transformers.__version__, "| adapters", adapters.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Use Runtime -> Change runtime type -> T4 GPU, then rerun from the top.")

torch 2.11.0+cu130 | transformers 4.57.6 | adapters 1.3.0
CUDA available: True
GPU: Tesla T4


In [ ]:
# ---------------- Configuration ----------------
BASE_MODEL = "FacebookAI/xlm-roberta-base"
EN_ADAPTER_ID = "AdapterHub/xlm-roberta-base-en-wiki_pfeiffer"
TA_ADAPTER_ID = "AdapterHub/xlm-roberta-base-ta-wiki_pfeiffer"
DATASET_ID = "dair-ai/emotion"
DATASET_CONFIG = "split"

try:
    from google.colab import drive
    print("Attempting to mount Google Drive...")
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/madx_tamil_emotion"   # final adapter + tokenizer (persists in Drive)
except Exception as e:
    print(f"Google Drive mount failed: {e}\nFalling back to local storage instead.")
    OUTPUT_DIR = "/content/madx_tamil_emotion"                 # Fallback local directory

CKPT_DIR = "/content/checkpoints"                          # temporary checkpoints (fast local disk)
os.makedirs(OUTPUT_DIR, exist_ok=True)
TASK_ADAPTER_NAME = "emotion_task"
MAX_LENGTH = 128
NUM_EPOCHS = 4
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-4
SEED = 42

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Attempting to mount Google Drive...
Mounted at /content/drive


## 1. English emotion data

In [ ]:
dataset = load_dataset(DATASET_ID, DATASET_CONFIG)
print(dataset)

# Take label names from the dataset itself so the order can never go out of sync.
LABEL_NAMES = dataset["train"].features["label"].names
ID2LABEL = {i: n for i, n in enumerate(LABEL_NAMES)}
LABEL2ID = {n: i for i, n in ID2LABEL.items()}
print("Labels:", LABEL_NAMES)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})
Labels: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized = dataset.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized = tokenized.rename_column("label", "labels")   # Trainer expects "labels"
collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2000
    })
})

## 2. Model, language adapters, task adapter

In [ ]:
model = AutoAdapterModel.from_pretrained(BASE_MODEL)

# source="hf" loads the adapters from the Hugging Face Hub.
en_name = model.load_adapter(EN_ADAPTER_ID, load_as="english", source="hf", with_head=False)
ta_name = model.load_adapter(TA_ADAPTER_ID, load_as="tamil", source="hf", with_head=False)
print("Loaded:", en_name, ta_name)

# New task adapter + 6-class head
model.add_adapter(TASK_ADAPTER_NAME, config="pfeiffer")
model.add_classification_head(
    TASK_ADAPTER_NAME,
    num_labels=len(LABEL_NAMES),
    id2label=ID2LABEL,
)

# Freeze everything except the task adapter and head
model.train_adapter(TASK_ADAPTER_NAME)

# Training-time stack: English language adapter -> emotion task adapter
# (must be set AFTER train_adapter, which resets the active setup)
model.set_active_adapters(Stack("english", TASK_ADAPTER_NAME))
model.active_head = TASK_ADAPTER_NAME

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.3f}%)")
print("Active adapters:", model.active_adapters)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaAdapterModel were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['heads.default.3.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

pytorch_adapter.bin:   0%|          | 0.00/29.6M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

Loaded: english tamil
Trainable parameters: 2,331,864 / 295,151,064 (0.790%)
Active adapters: Stack[english, emotion_task]


## 3. Train the English task adapter

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }

training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    remove_unused_columns=False,     # keep columns intact for the adapter model
    seed=SEED,
)

# AdapterTrainer (not Trainer) saves/loads only the adapter weights in checkpoints.
trainer = AdapterTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.365900,0.290902,0.906500,0.882855,0.907211
2,0.219300,0.246331,0.920500,0.894000,0.920717
3,0.173200,0.193535,0.931500,0.905285,0.931425
4,0.143700,0.185894,0.929500,0.904360,0.929527


TrainOutput(global_step=4000, training_loss=0.3298544223308563, metrics={'train_runtime': 411.2333, 'train_samples_per_second': 155.629, 'train_steps_per_second': 9.727, 'total_flos': 2017138448054016.0, 'train_loss': 0.3298544223308563, 'epoch': 4.0})

In [ ]:
print("English validation:", trainer.evaluate(tokenized["validation"]))
print("English test:      ", trainer.evaluate(tokenized["test"]))

English validation: {'eval_loss': 0.19353508949279785, 'eval_accuracy': 0.9315, 'eval_macro_f1': 0.9052851725247791, 'eval_weighted_f1': 0.9314249063285372, 'eval_runtime': 2.707, 'eval_samples_per_second': 738.834, 'eval_steps_per_second': 23.273, 'epoch': 4.0}
English test:       {'eval_loss': 0.1975126564502716, 'eval_accuracy': 0.926, 'eval_macro_f1': 0.8826124217793918, 'eval_weighted_f1': 0.9253188512270215, 'eval_runtime': 2.7884, 'eval_samples_per_second': 717.257, 'eval_steps_per_second': 22.594, 'epoch': 4.0}


In [ ]:
# Save the task adapter (with its head) and tokenizer
task_adapter_dir = os.path.join(OUTPUT_DIR, "emotion_task_adapter")
os.makedirs(task_adapter_dir, exist_ok=True)
model.save_adapter(task_adapter_dir, TASK_ADAPTER_NAME, with_head=True)
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "tokenizer"))
print("Saved to:", task_adapter_dir)

Saved to: /content/drive/MyDrive/madx_tamil_emotion/emotion_task_adapter


## 4. Swap in the Tamil language adapter (zero-shot)

In [ ]:
# Task adapter stays unchanged; only the language adapter changes.
model.set_active_adapters(Stack("tamil", TASK_ADAPTER_NAME))
model.active_head = TASK_ADAPTER_NAME
model.to(device)
model.eval()
print("Active adapters:", model.active_adapters)

def predict_proba(texts, allowed=None, batch_size=32):
    """Return an (N, num_labels) probability array.
    allowed: optional list of label names; all other classes are masked out."""
    mask = None
    if allowed is not None:
        mask = torch.full((len(LABEL_NAMES),), float("-inf"))
        for l in allowed:
            mask[LABEL2ID[l]] = 0.0
        mask = mask.to(device)

    out = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_LENGTH).to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        if mask is not None:
            logits = logits + mask
        out.append(torch.softmax(logits, dim=-1).cpu())
    return torch.cat(out).numpy()

def predict_emotion(text, allowed=None):
    p = predict_proba([text], allowed=allowed)[0]
    best = int(p.argmax())
    return {
        "text": text,
        "predicted_emotion": ID2LABEL[best],
        "confidence": float(p[best]),
        "all_scores": {ID2LABEL[i]: round(float(p[i]), 4) for i in range(len(LABEL_NAMES))},
    }

examples = [
    "இன்று எனக்கு மிகவும் மகிழ்ச்சியாக இருக்கிறது.",
    "அவன் செய்ததை நினைத்தாலே எனக்கு கோபமாக இருக்கிறது.",
    "என் நண்பரைப் பிரிந்ததால் மிகவும் வருத்தமாக இருக்கிறது.",
    "நாளைய முடிவைப் பற்றி எனக்குப் பயமாக இருக்கிறது.",
    "இந்தச் செய்தியை நம்பவே முடியவில்லை!",
]

for s in examples:
    r = predict_emotion(s)
    print(f"\n{r['text']}\n-> {r['predicted_emotion']} ({r['confidence']:.3f})  {r['all_scores']}")

Active adapters: Stack[tamil, emotion_task]

இன்று எனக்கு மிகவும் மகிழ்ச்சியாக இருக்கிறது.
-> joy (0.999)  {'sadness': 0.0005, 'joy': 0.9993, 'love': 0.0, 'anger': 0.0001, 'fear': 0.0, 'surprise': 0.0}

அவன் செய்ததை நினைத்தாலே எனக்கு கோபமாக இருக்கிறது.
-> anger (0.993)  {'sadness': 0.0016, 'joy': 0.0001, 'love': 0.0, 'anger': 0.9928, 'fear': 0.0053, 'surprise': 0.0}

என் நண்பரைப் பிரிந்ததால் மிகவும் வருத்தமாக இருக்கிறது.
-> sadness (0.947)  {'sadness': 0.9475, 'joy': 0.0011, 'love': 0.0001, 'anger': 0.0508, 'fear': 0.0005, 'surprise': 0.0}

நாளைய முடிவைப் பற்றி எனக்குப் பயமாக இருக்கிறது.
-> fear (0.925)  {'sadness': 0.0055, 'joy': 0.0003, 'love': 0.0, 'anger': 0.0686, 'fear': 0.9254, 'surprise': 0.0002}

இந்தச் செய்தியை நம்பவே முடியவில்லை!
-> fear (0.983)  {'sadness': 0.002, 'joy': 0.0036, 'love': 0.0, 'anger': 0.0031, 'fear': 0.9834, 'surprise': 0.0079}


## 5. Evaluate on a labelled Tamil test set

Expected CSV columns: `text`, `label`. Labels must be lowercase names from the English label set (`sadness, joy, love, anger, fear, surprise`).

If your Tamil data uses fewer classes (e.g. joy / sadness / anger / fear), pass them in `ALLOWED` so the English-only classes (`love`, `surprise`) cannot be predicted.

In [6]:
import pandas as pd
import os

# Point this directly to the file you uploaded to Google Drive in Step 1
TAMIL_CSV = "/content/drive/MyDrive/Tamil_Emotions.csv"

ALLOWED = None                         # e.g. ["joy", "sadness", "anger", "fear"]

def evaluate_tamil_test(texts, gold_labels, allowed=None):
    unknown = set(gold_labels) - set(LABEL2ID)
    if unknown:
        raise ValueError(f"Unknown labels {unknown}; expected a subset of {LABEL_NAMES}")

    probs = predict_proba(list(texts), allowed=allowed)
    y_pred = probs.argmax(axis=1)
    y_true = np.array([LABEL2ID[g] for g in gold_labels])

    present = sorted(set(y_true) | set(y_pred))
    print("Accuracy :", round(accuracy_score(y_true, y_pred), 4))
    print("Macro-F1 :", round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4))
    print(classification_report(
        y_true, y_pred, labels=present,
        target_names=[ID2LABEL[i] for i in present], zero_division=0))
    return y_true, y_pred

if TAMIL_CSV and os.path.exists(TAMIL_CSV):
    # Fix: Check the file extension and use the correct Pandas reader
    if TAMIL_CSV.endswith('.xlsx'):
        df = pd.read_excel(TAMIL_CSV)
    else:
        df = pd.read_csv(TAMIL_CSV)

    df["label"] = df["label"].astype(str).str.strip().str.lower()
    y_true, y_pred = evaluate_tamil_test(df["text"].astype(str).tolist(), df["label"].tolist(), allowed=ALLOWED)
else:
    print(f"Error: Could not find the file at {TAMIL_CSV}. Please check your Google Drive.")

Error: Could not find the file at /content/drive/MyDrive/Tamil_Emotions.csv. Please check your Google Drive.


In [17]:
# 0. Automatically reinstall packages with strict compatible versions if lost due to session restart
try:
    import sys
    import adapters
    import datasets
    from huggingface_hub import HfFolder
except (ImportError, NameError):
    print("Dependencies missing or incompatible. Forcing install of compatible huggingface_hub & adapters...")
    !pip install -q "huggingface-hub<2.0" adapters datasets accelerate sentencepiece

    # Force Python to reload the freshly installed packages by clearing module cache
    import sys
    for module_name in list(sys.modules.keys()):
        if "huggingface_hub" in module_name or "adapters" in module_name or "datasets" in module_name:
            sys.modules.pop(module_name, None)

import pandas as pd
import numpy as np
import os
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from adapters import AutoAdapterModel
from adapters.composition import Stack
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. Re-initialize label mappings if missing
try:
    _ = LABEL_NAMES
    _ = LABEL2ID
    _ = ID2LABEL
except NameError:
    print("Initializing dataset label mappings...")
    dataset = load_dataset("dair-ai/emotion", "split")
    LABEL_NAMES = dataset["train"].features["label"].names
    ID2LABEL = {i: n for i, n in enumerate(LABEL_NAMES)}
    LABEL2ID = {n: i for i, n in ID2LABEL.items()}

# 2. Re-initialize model, tokenizer, and prediction function if missing
BASE_MODEL = "FacebookAI/xlm-roberta-base"
OUTPUT_DIR = "/content/drive/MyDrive/madx_tamil_emotion"
TASK_ADAPTER_NAME = "emotion_task"
MAX_LENGTH = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    _ = predict_proba
    print("predict_proba is already loaded.")
except NameError:
    print("Loading tokenizer and trained adapter model from Google Drive...")
    tokenizer = AutoTokenizer.from_pretrained(os.path.join(OUTPUT_DIR, "tokenizer"))
    model = AutoAdapterModel.from_pretrained(BASE_MODEL)

    # Load language adapters and task adapter
    model.load_adapter("AdapterHub/xlm-roberta-base-ta-wiki_pfeiffer", load_as="tamil", source="hf", with_head=False)
    task_adapter_path = os.path.join(OUTPUT_DIR, "emotion_task_adapter")
    model.load_adapter(task_adapter_path, load_as=TASK_ADAPTER_NAME, with_head=True)

    # Set active configuration
    model.set_active_adapters(Stack("tamil", TASK_ADAPTER_NAME))
    model.active_head = TASK_ADAPTER_NAME
    model.to(device)
    model.eval()

    def predict_proba(texts, allowed=None, batch_size=32):
        mask = None
        if allowed is not None:
            mask = torch.full((len(LABEL_NAMES),), float("-inf"))
            for l in allowed:
                mask[LABEL2ID[l]] = 0.0
            mask = mask.to(device)

        out = []
        for i in range(0, len(texts), batch_size):
            enc = tokenizer(texts[i:i + batch_size], return_tensors="pt", padding=True,
                            truncation=True, max_length=MAX_LENGTH).to(device)
            with torch.no_grad():
                logits = model(**enc).logits
            if mask is not None:
                logits = logits + mask
            out.append(torch.softmax(logits, dim=-1).cpu())
        return torch.cat(out).numpy()

# 3. Define the evaluation function
def evaluate_tamil_test(texts, gold_labels, allowed=None):
    unknown = set(gold_labels) - set(LABEL2ID)
    if unknown:
        raise ValueError(f"Unknown labels {unknown}; expected a subset of {LABEL_NAMES}")

    probs = predict_proba(list(texts), allowed=allowed)
    y_pred = probs.argmax(axis=1)
    y_true = np.array([LABEL2ID[g] for g in gold_labels])

    present = sorted(set(y_true) | set(y_pred))
    print("Accuracy :", round(accuracy_score(y_true, y_pred), 4))
    print("Macro-F1 :", round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4))
    print(classification_report(
        y_true, y_pred, labels=present,
        target_names=[ID2LABEL[i] for i in present], zero_division=0))
    return y_true, y_pred

# 4. Load & preprocess CSV and run evaluation
TAMIL_CSV = "/content/drive/MyDrive/Tamil_Emotions.csv"

if os.path.exists(TAMIL_CSV):
    df = pd.read_csv(TAMIL_CSV)
    print(f"\nLoaded dataset with {len(df)} rows.")

    # Normalize labels
    df["label"] = df["label"].astype(str).str.strip().str.lower()

    # Map custom labels
    label_mapping = {
        "happy": "joy",
        "disgust": "anger"
    }
    df["label"] = df["label"].replace(label_mapping)

    # Filter unrecognized labels
    valid_labels = set(LABEL_NAMES)
    initial_count = len(df)
    df = df[df["label"].isin(valid_labels)]
    if len(df) < initial_count:
        print(f"Filtered out {initial_count - len(df)} rows with unrecognized labels.")

    # Run evaluation
    y_true, y_pred = evaluate_tamil_test(
        df["text"].astype(str).tolist(),
        df["label"].tolist(),
        allowed=None
    )
else:
    print(f"Error: Still unable to access {TAMIL_CSV}")

Dependencies missing or incompatible. Forcing install of compatible huggingface_hub & adapters...


/usr/local/lib/python3.13/dist-packages/datasets/filesystems/__init__.py:24: UserWarning: A filesystem protocol was already set for bz2 and will be overwritten.
  warnings.warn(f"A filesystem protocol was already set for {fs_class.protocol} and will be overwritten.")
/usr/local/lib/python3.13/dist-packages/datasets/filesystems/__init__.py:24: UserWarning: A filesystem protocol was already set for gzip and will be overwritten.
  warnings.warn(f"A filesystem protocol was already set for {fs_class.protocol} and will be overwritten.")
/usr/local/lib/python3.13/dist-packages/datasets/filesystems/__init__.py:24: UserWarning: A filesystem protocol was already set for lz4 and will be overwritten.
  warnings.warn(f"A filesystem protocol was already set for {fs_class.protocol} and will be overwritten.")
/usr/local/lib/python3.13/dist-packages/datasets/filesystems/__init__.py:24: UserWarning: A filesystem protocol was already set for xz and will be overwritten.
  warnings.warn(f"A filesystem prot

ArrowKeyError: A type extension with name datasets.features.features.Array2DExtensionType already defined

In [9]:
from google.colab import drive, files
import os

try:
    print("Attempting to mount Google Drive...")
    drive.mount("/content/drive", force_remount=True)

    drive_path = "/content/drive/MyDrive"
    if os.path.exists(drive_path):
        print("\nGoogle Drive successfully mounted. Files found:")
        found_files = [f for f in os.listdir(drive_path) if f.endswith(('.csv', '.xlsx', '.xls', '.txt')) or 'tamil' in f.lower()]
        for f in sorted(found_files):
            print(f"- {f}")
    else:
        print("\nDrive folder path not found after mount.")
except Exception as e:
    print(f"\nDrive mount failed: {e}")
    print("Alternative: If you want to upload the file directly from your local computer, run the code below:")
    # uploaded = files.upload()

Attempting to mount Google Drive...
Mounted at /content/drive

Google Drive successfully mounted. Files found:
- 04-02-26-111-MSUC-Attendnce template-Data Analytics and Visualization using apex.xlsx
- 06-02-26-111-MSUC-Attendnce template-Data Analytics and Visualization using apex.xlsx
- 06-03-26-111-MSUC-Attendnce template-Data Analytics and Visualization using apex.xlsx
- 11-02-26-111-MSUC-Attendnce template-Data Analytics and Visualization using apex.xlsx
- 11-03-26-111-MSUC-Attendnce template-Data Analytics and Visualization using apex.xlsx
- 13-02-26-111-MSUC-Attendnce template-Data Analytics and Visualization using apex.xlsx
- 13-03-26-111-MSUC-Attendnce template-Data Analytics and Visualization using apex.xlsx
- 18-02-26-111-MSUC-Attendnce template-Data Analytics and Visualization using apex.xlsx
- 18-03-26-111-MSUC-Attendnce template-Data Analytics and Visualization using apex.xlsx
- 20-02-26-111-MSUC-Attendnce template-Data Analytics and Visualization using apex.xlsx
- 21-01-2

In [1]:
# Clean dependency setup cell to install fully compatible versions of adapters
print("Upgrading dependencies to resolve huggingface_hub integration issues...")
!pip install -q -U adapters huggingface_hub transformers datasets accelerate sentencepiece
print("Installation complete. Please run this cell, click 'Runtime' -> 'Restart session' to apply the new library configurations, then execute your evaluation cell.")

Upgrading dependencies to resolve huggingface_hub integration issues...
Installation complete. Please run this cell, click 'Runtime' -> 'Restart session' to apply the new library configurations, then execute your evaluation cell.


In [1]:
import pandas as pd
import numpy as np
import os
import torch
from transformers import AutoTokenizer
from adapters import AutoAdapterModel
from adapters.composition import Stack
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. Statically define the label mappings to completely bypass importing 'datasets' package (and avoid ArrowKeyError)
LABEL_NAMES = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
ID2LABEL = {i: n for i, n in enumerate(LABEL_NAMES)}
LABEL2ID = {n: i for i, n in ID2LABEL.items()}
print("Using defined dataset label mappings:", LABEL_NAMES)

# 2. Re-initialize model, tokenizer, and prediction function if missing
BASE_MODEL = "FacebookAI/xlm-roberta-base"
OUTPUT_DIR = "/content/drive/MyDrive/madx_tamil_emotion"
TASK_ADAPTER_NAME = "emotion_task"
MAX_LENGTH = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    _ = predict_proba
    print("predict_proba is already loaded.")
except NameError:
    print("Loading tokenizer and trained adapter model from Google Drive...")
    tokenizer = AutoTokenizer.from_pretrained(os.path.join(OUTPUT_DIR, "tokenizer"))
    model = AutoAdapterModel.from_pretrained(BASE_MODEL)

    # Load language adapters and task adapter
    model.load_adapter("AdapterHub/xlm-roberta-base-ta-wiki_pfeiffer", load_as="tamil", source="hf", with_head=False)
    task_adapter_path = os.path.join(OUTPUT_DIR, "emotion_task_adapter")
    model.load_adapter(task_adapter_path, load_as=TASK_ADAPTER_NAME, with_head=True)

    # Set active configuration
    model.set_active_adapters(Stack("tamil", TASK_ADAPTER_NAME))
    model.active_head = TASK_ADAPTER_NAME
    model.to(device)
    model.eval()

    def predict_proba(texts, allowed=None, batch_size=32):
        mask = None
        if allowed is not None:
            mask = torch.full((len(LABEL_NAMES),), float("-inf"))
            for l in allowed:
                mask[LABEL2ID[l]] = 0.0
            mask = mask.to(device)

        out = []
        for i in range(0, len(texts), batch_size):
            enc = tokenizer(texts[i:i + batch_size], return_tensors="pt", padding=True,
                            truncation=True, max_length=MAX_LENGTH).to(device)
            with torch.no_grad():
                logits = model(**enc).logits
            if mask is not None:
                logits = logits + mask
            out.append(torch.softmax(logits, dim=-1).cpu())
        return torch.cat(out).numpy()

# 3. Define the evaluation function
def evaluate_tamil_test(texts, gold_labels, allowed=None):
    unknown = set(gold_labels) - set(LABEL2ID)
    if unknown:
        raise ValueError(f"Unknown labels {unknown}; expected a subset of {LABEL_NAMES}")

    probs = predict_proba(list(texts), allowed=allowed)
    y_pred = probs.argmax(axis=1)
    y_true = np.array([LABEL2ID[g] for g in gold_labels])

    present = sorted(set(y_true) | set(y_pred))
    print("Accuracy :", round(accuracy_score(y_true, y_pred), 4))
    print("Macro-F1 :", round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4))
    print(classification_report(
        y_true, y_pred, labels=present,
        target_names=[ID2LABEL[i] for i in present], zero_division=0))
    return y_true, y_pred

# 4. Load & preprocess CSV and run evaluation
TAMIL_CSV = "/content/drive/MyDrive/Tamil_Emotions.csv"

if os.path.exists(TAMIL_CSV):
    df = pd.read_csv(TAMIL_CSV)
    print(f"\nLoaded dataset with {len(df)} rows.")

    # Normalize labels
    df["label"] = df["label"].astype(str).str.strip().str.lower()

    # Map custom labels
    label_mapping = {
        "happy": "joy",
        "disgust": "anger"
    }
    df["label"] = df["label"].replace(label_mapping)

    # Filter unrecognized labels
    valid_labels = set(LABEL_NAMES)
    initial_count = len(df)
    df = df[df["label"].isin(valid_labels)]
    if len(df) < initial_count:
        print(f"Filtered out {initial_count - len(df)} rows with unrecognized labels.")

    # Run evaluation
    y_true, y_pred = evaluate_tamil_test(
        df["text"].astype(str).tolist(),
        df["label"].tolist(),
        allowed=None
    )
else:
    print(f"Error: Still unable to access {TAMIL_CSV}")

Using defined dataset label mappings: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
Loading tokenizer and trained adapter model from Google Drive...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaAdapterModel were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['heads.default.3.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

pytorch_adapter.bin:   0%|          | 0.00/29.6M [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]


Loaded dataset with 1027 rows.
Accuracy : 0.295
Macro-F1 : 0.0946
              precision    recall  f1-score   support

     sadness       0.00      0.00      0.00       191
         joy       0.35      0.04      0.08       182
        love       0.00      0.00      0.00         0
       anger       0.30      0.98      0.45       298
        fear       0.14      0.01      0.03       143
    surprise       1.00      0.00      0.01       213

    accuracy                           0.30      1027
   macro avg       0.30      0.17      0.09      1027
weighted avg       0.37      0.30      0.15      1027

